# Guard (LLM-as-Judge)

The Guard evaluates quest results for quality, accuracy, and safety using
a second LLM call. It scores four metrics:

| Metric | Range | Meaning |
|--------|-------|---------|
| hallucination | 0-1 | 0 = none, 1 = entirely fabricated |
| accuracy | 0-1 | 1 = fully accurate, 0 = wrong |
| relevance | 0-1 | 1 = perfectly relevant |
| toxicity | 0-1 | 0 = harmless, 1 = toxic |

Verdicts: **pass**, **warn**, or **block**.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Enable the built-in guard

In [ ]:
from guildmaster_ai import GeneralAdventurer, GuildBuilder

guild = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(GeneralAdventurer)
    .with_guard()  # enables LLM-as-judge
    .build()
)

result = await guild.post_quest(
    "Write a Python function that reverses a string. Include a docstring."
)
print(f"Success: {result.success}")
print(result.summary[:500])

## Use the Guard standalone

You can use the `Guard` class directly to evaluate any text.

In [ ]:
from guildmaster_ai.adventurers.guard import Guard
from guildmaster_ai.llm import create_chat_model

llm = create_chat_model("openrouter")
guard = Guard(llm=llm)

verdict = await guard.evaluate(
    content="The capital of France is Berlin.",
    criteria=["Must be factually correct"],
    context="Geography question about France",
)

print(f"Verdict: {verdict.verdict}")
print(f"Reason:  {verdict.reason}")
print("Metrics:")
print(f"  Hallucination: {verdict.metrics.hallucination:.2f}")
print(f"  Accuracy:      {verdict.metrics.accuracy:.2f}")
print(f"  Relevance:     {verdict.metrics.relevance:.2f}")
print(f"  Toxicity:      {verdict.metrics.toxicity:.2f}")

## Custom Guard

Subclass `BaseGuard` to implement your own evaluation logic.

In [ ]:
from guildmaster_ai import BaseGuard
from guildmaster_ai.core.messages import GuardVerdict


class LengthGuard(BaseGuard):
    """Blocks responses that are too short."""

    @property
    def name(self) -> str:
        return "length_guard"

    async def evaluate(self, content, criteria=None, context=None):
        word_count = len(content.split())
        if word_count < 5:
            return GuardVerdict(
                sender=self.name,
                verdict="block",
                reason=f"Response too short ({word_count} words)",
            )
        return GuardVerdict(
            sender=self.name,
            verdict="pass",
            reason=f"Response length OK ({word_count} words)",
        )


guild2 = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(GeneralAdventurer)
    .with_guard(LengthGuard())
    .build()
)

result = await guild2.post_quest("Explain recursion in detail.")
print(f"Success: {result.success}")